# 02 — Análise estatística

Testes para responder, com incerteza explícita:
1. A tendência de custo é consistente? (Theil–Sen + Mann–Kendall)
2. Quais categorias variam mais entre UFs? (CV, P90/P10)
3. Há diferença entre regiões dentro do mesmo subgrupo? (Kruskal–Wallis, ε², Holm)
4. Volume e custo estão associados? (Spearman) — **associação, não causalidade**

> ⚠️ Base sintética — resultados ilustrativos.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from hcb.config import load_settings
from hcb.analysis import indicators, statistics, outliers
from hcb.pipeline import FACT_FILE, run
from hcb.schema import REGION_ORDER

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9", "axes.titlelocation": "left"})
REGION_COLORS = {"Norte": "#2a78d6", "Nordeste": "#eb6834", "Centro-Oeste": "#1baf7a",
                 "Sudeste": "#eda100", "Sul": "#e87ba4"}

settings = load_settings()
if not (settings.processed_dir / FACT_FILE).exists():
    run(settings)
fact = pd.read_csv(settings.processed_dir / FACT_FILE, dtype={"subgrupo_codigo": str, "grupo_codigo": str},
                   parse_dates=["data"])
fact["regiao"] = pd.Categorical(fact["regiao"], categories=REGION_ORDER, ordered=True)
print(f"{len(fact):,} linhas | {fact['competencia'].min()} a {fact['competencia'].max()}")


14,992 linhas | 2022-01 a 2024-12


## 1. Tendência
Theil–Sen estima a inclinação mediana entre todos os pares de pontos — robusta a meses atípicos. Aplicada ao
log do custo médio mensal, a inclinação vira taxa de crescimento composta anual.

In [2]:
statistics.trends_by(fact, "subgrupo_nome")[["subgrupo_nome", "crescimento_anual", "ic95_inf", "ic95_sup",
                                             "p_ajustado_holm", "tendencia"]]

,subgrupo_nome,crescimento_anual,ic95_inf,ic95_sup,p_ajustado_holm,tendencia
0,Cirurgia do aparelho da visão,0.06,0.06,0.07,0.00,alta
1,Tratamentos clínicos (outras especialidades),0.06,0.06,0.06,0.00,alta
2,Cirurgia em oncologia,0.06,0.06,0.06,0.00,alta
3,Outras cirurgias,0.06,0.06,0.06,0.00,alta
4,Parto e nascimento,0.06,0.06,0.06,0.00,alta
5,Cirurgia do aparelho circulatório,0.06,0.06,0.06,0.00,alta
6,Tratamento em oncologia,0.06,0.06,0.06,0.00,alta
7,Cirurgia do aparelho geniturinário,0.06,0.06,0.06,0.00,alta
8,Cirurgia do sistema osteomuscular,0.06,0.06,0.07,0.00,alta
9,Cirurgia obstétrica,0.06,0.06,0.06,0.00,alta


## 2. Variação entre UFs por subgrupo

In [3]:
statistics.variation_between_ufs(fact)

,subgrupo_codigo,subgrupo_nome,ufs_avaliadas,custo_mediano_ufs,cv_entre_ufs,razao_p90_p10,uf_menor_custo,uf_maior_custo,menor_custo,maior_custo
0,0406,Cirurgia do aparelho circulatório,27,"8,181.57",0.12,1.38,CE,SP,"7,467.34","11,252.26"
1,0308,"Tratamento de lesões, envenenamentos e outros,...",27,973.78,0.12,1.35,BA,DF,814.78,"1,199.79"
2,0408,Cirurgia do sistema osteomuscular,27,"2,241.03",0.12,1.34,RN,PR,"2,015.01","2,814.03"
3,0304,Tratamento em oncologia,27,"2,434.68",0.11,1.30,MA,ES,"2,079.28","3,150.37"
4,0416,Cirurgia em oncologia,27,"4,095.11",0.11,1.34,PI,AC,"3,318.71","4,830.41"
5,0403,Cirurgia do sistema nervoso central e periférico,27,"5,286.82",0.10,1.28,BA,SP,"4,772.62","6,924.29"
6,0404,"Cirurgia das vias aéreas superiores, da face, ...",27,944.87,0.10,1.30,PI,ES,804.02,"1,150.62"
7,0415,Outras cirurgias,27,"3,757.21",0.10,1.28,RN,RS,"3,203.69","4,673.14"
8,0411,Cirurgia obstétrica,27,906.64,0.10,1.29,PE,RJ,737.59,"1,129.02"
9,0303,Tratamentos clínicos (outras especialidades),27,"1,186.82",0.10,1.29,RN,PR,"1,036.48","1,397.03"


## 3. Diferenças regionais
Unidade = UF no período inteiro (n ≤ 27 por subgrupo). Usar UF × mês inflaria artificialmente o n com meses
correlacionados (pseudo-replicação). ε² ≥ 0,26 é considerado efeito grande.

In [4]:
statistics.regional_differences(fact)[["subgrupo_nome", "ufs", "h", "p_ajustado_holm", "epsilon2",
                                       "magnitude_efeito", "regiao_maior_mediana", "regiao_menor_mediana"]]

,subgrupo_nome,ufs,h,p_ajustado_holm,epsilon2,magnitude_efeito,regiao_maior_mediana,regiao_menor_mediana
0,Tratamentos clínicos (outras especialidades),27,18.69,0.01,0.72,grande,Sul,Nordeste
1,Outras cirurgias,27,17.97,0.02,0.69,grande,Sul,Nordeste
2,Tratamento em oncologia,27,17.54,0.02,0.67,grande,Sul,Nordeste
3,"Tratamento de lesões, envenenamentos e outros,...",27,16.13,0.04,0.62,grande,Sul,Nordeste
4,Parto e nascimento,27,15.07,0.05,0.58,grande,Sul,Norte
5,"Transplante de órgãos, tecidos e células",12,6.37,0.13,0.58,grande,Sudeste,Nordeste
6,"Cirurgia das vias aéreas superiores, da face, ...",27,14.98,0.05,0.58,grande,Sudeste,Nordeste
7,Cirurgia obstétrica,27,14.46,0.06,0.56,grande,Sudeste,Nordeste
8,Cirurgia do sistema nervoso central e periférico,27,14.32,0.06,0.55,grande,Sul,Nordeste
9,Tratamento em nefrologia,27,14.00,0.06,0.54,grande,Sul,Nordeste


## 4. Volume × custo (Spearman)

In [5]:
statistics.spearman_volume_cost(fact).sort_values('rho_spearman')

,subgrupo_codigo,subgrupo_nome,ufs,rho_spearman,p_valor,p_ajustado_holm
15,0505,"Transplante de órgãos, tecidos e células",12,-0.17,0.59,1.00
3,0308,"Tratamento de lesões, envenenamentos e outros,...",27,0.12,0.55,1.00
10,0408,Cirurgia do sistema osteomuscular,27,0.20,0.31,0.94
5,0403,Cirurgia do sistema nervoso central e periférico,27,0.29,0.14,0.72
6,0404,"Cirurgia das vias aéreas superiores, da face, ...",27,0.30,0.13,0.72
2,0305,Tratamento em nefrologia,27,0.31,0.12,0.72
9,0407,"Cirurgia do aparelho digestivo, órgãos anexos ...",27,0.35,0.07,0.59
13,0415,Outras cirurgias,27,0.35,0.07,0.59
11,0409,Cirurgia do aparelho geniturinário,27,0.36,0.07,0.59
0,0303,Tratamentos clínicos (outras especialidades),27,0.37,0.06,0.58


**Cuidados de interpretação.** Uma correlação entre volume e custo pode refletir perfil de casos
(centros de referência recebem casos mais graves), estrutura de rede, tabelas complementares estaduais ou
diferenças de registro. Nenhuma das análises acima permite afirmar que *aumentar* ou *reduzir* volume
alteraria o custo.